In [10]:
from PIL import Image
import numpy as np

def image_to_tensor(filepath, dtype=np.float32, normalize=True):
    # Open image and convert to RGB
    image = Image.open(filepath).convert('RGB')  # ensures 3 channels

    # Convert to NumPy array (H, W, C)
    img_np = np.array(image, dtype=dtype)

    if normalize:
        img_np /= 255.0  # normalize to [0, 1]

    # Transpose to (C, H, W)
    img_np = np.transpose(img_np, (2, 0, 1))

    # Add batch dimension -> (1, C, H, W)
    img_tensor = np.expand_dims(img_np, axis=0)

    return img_tensor  # shape: [1, C, H, W]





In [11]:
#image_path="570216.png"
image_path="1920x1080-full-hd-nature-landscape.jpg"
tensor = image_to_tensor(image_path)
np.save("x.npy", tensor)

In [12]:
import numpy as np

channels = 3
kernel_size = 3

# Luminance coefficients for RGB
luminance = np.array([0.2989, 0.5870, 0.1140], dtype=np.float32)

# Create empty kernel: shape [1, channels, kernel_size, kernel_size]
mean_kernel = np.zeros((1, channels, kernel_size, kernel_size), dtype=np.float32)

# Fill each channel with its luminance spread over the kernel
for c in range(channels):
    mean_kernel[0, c, :, :] = luminance[c] / (kernel_size ** 2)

# mean_kernel is now ready to use
print("Kernel shape:", mean_kernel.shape)
print(mean_kernel)
np.save("w.npy",mean_kernel)

Kernel shape: (1, 3, 3, 3)
[[[[0.03321111 0.03321111 0.03321111]
   [0.03321111 0.03321111 0.03321111]
   [0.03321111 0.03321111 0.03321111]]

  [[0.06522223 0.06522223 0.06522223]
   [0.06522223 0.06522223 0.06522223]
   [0.06522223 0.06522223 0.06522223]]

  [[0.01266667 0.01266667 0.01266667]
   [0.01266667 0.01266667 0.01266667]
   [0.01266667 0.01266667 0.01266667]]]]


In [18]:
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# Load inputs
x = np.load("x.npy").astype(np.float32)  # Shape: [N, C, H, W]
w = np.load("w.npy").astype(np.float32)  # Shape: [M, C, kH, kW]

# Optional: You can also provide a bias (shape = [M])
# bias = np.zeros((w.shape[0],), dtype=np.float32)

# Create input & output value infos
input_tensor = helper.make_tensor_value_info("x", TensorProto.FLOAT, list(x.shape))
weight_tensor = helper.make_tensor_value_info("w", TensorProto.FLOAT, list(w.shape))
output_tensor = helper.make_tensor_value_info("y", TensorProto.FLOAT, None)

# Create Conv node
node_with_padding = helper.make_node(
    "Conv",
    inputs=["x", "w"],  # optionally add "b" if using bias
    outputs=["y"],
    kernel_shape=[3, 3],
    pads=[1, 1, 1, 1],
    strides=[2, 2],
)

# Define the graph
graph_def = helper.make_graph(
    nodes=[node_with_padding],
    name="ConvModel",
    inputs=[input_tensor, weight_tensor],
    outputs=[output_tensor]
)

# Create the model
model = helper.make_model(graph_def, producer_name="onnx-conv-example")

# Run inference
session = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
inputs = {
    "x": x,
    "w": w,
}
output = session.run(["y"], inputs)
print("Output shape:", output[0].shape)
#print(output[0])


Output shape: (1, 1, 540, 960)


In [28]:
#import numpy as np
#from PIL import Image
from numpy.testing import assert_allclose 

# Assuming `output` is the list returned from session.run(...)
output_tensor = output[0]  # Shape: [1, C, H, W]
output_tensor = np.load("y_cpp.npy")
print("Output shape:", output_tensor.shape)
assert_allclose(output[0], output_tensor,rtol=1e-6,atol=1e-7)
output_tensor = np.squeeze(output_tensor)  # Remove batch dim → [C, H, W]

# If single-channel (grayscale)
if output_tensor.ndim == 2:
    img = Image.fromarray((output_tensor * 255).astype(np.uint8), mode='L')
    img.save("output.png")

# If 3 channels (RGB)
elif output_tensor.shape[0] == 3:
    img = np.transpose(output_tensor, (1, 2, 0))  # CHW → HWC
    img = (img * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(img).save("output.png")

# If more than 3 channels (e.g., feature maps), visualize first channel
else:
    first_channel = output_tensor[0]
    img = Image.fromarray((first_channel * 255).astype(np.uint8), mode='L')
    img.save("output_first_channel.png")

Output shape: (1, 1, 540, 960)
